# 3.2 — Swin Transformer fine-tuning completo — Food-101

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcoslund/ViT-for-101-food-app/blob/main/notebooks/3.2-swin.ipynb)

Entrenamiento completo de **Swin Transformer** (`microsoft/swin-tiny-patch4-window7-224`) con la estructura del proyecto y los artefactos de `2.0-preprocessing.ipynb`.

Swin es un ViT **jerarquico**: aplica self-attention en **ventanas locales** (W-MSA) y las va **desplazando** entre capas (SW-MSA) para conectar ventanas vecinas, con *patch merging* que reduce resolucion y agranda el campo receptivo por etapa, como una CNN (teoria: Clase 3 del material CEIA-ViT).

- El **codigo de modelo y entrenamiento vive en `modeling/training.py`** y la orquestacion en `modeling/benchmark.py`; aca solo se llaman funciones.
- Reusa `config.py` y `preprocessing/loaders.py`: no se reimplementan resize, crop, normalizacion ni augmentation.
- La evaluacion usa `modeling/evaluation.py`, **el mismo codigo para todos los modelos del benchmark**.
- `test` se usa una sola vez al final; la pregunta del proyecto la contesta el **subset del benchmark por tercil**.
- Al terminar, guarda el mejor modelo y lo **descarga a tu computadora** (sección 8.1).
- Checkpoints bajo `models/swin/full/`; resultados livianos en `reports/results/swin/`.

**Variante:** swin-tiny (~28M params, 224px) es el punto intermedio entre MobileViT-small (~5M) y ViT-base (~86M), y entrena en pocas horas en una T4. Para mas precision a costa de ~1.7x tiempo, cambiar la linea `swin` de `config.MODELS` a `microsoft/swin-small-patch4-window7-224`. La notebook lee la resolucion del checkpoint y ajusta el batch sola.

> Para una corrida **desatendida** (Kaggle, VM), el mismo pipeline esta como CLI: `python -m vit_for_101_food_app.modeling.train --model swin`.

## 0. Antes de ejecutar

Desde la raiz del repo:

```bash
uv sync --extra deep
uv run jupyter lab
```

El preprocessing debe existir (`make preprocess` o `notebooks/2.0-preprocessing.ipynb`). En **Colab** no hace falta nada previo: sección 0.2 y sección 0.3 arman todo.

## 0.1 Parametros de ejecucion

- `USE_DRIVE` (solo Colab): guarda dataset, checkpoints y resultados en Google Drive. El disco de Colab se borra al desconectarse; sin Drive, una desconexion a mitad de entrenamiento pierde todo.
- `RESUME`: `True` retoma desde el ultimo checkpoint de `OUTPUT_DIR` (corrida cortada). Con `False` y checkpoints viejos, la notebook se detiene en vez de mezclarlos.
- `REPO_BRANCH`: rama que Colab clona. Con todo mergeado a `main`, dejar `"main"`.

In [ ]:
# Solo Colab: persistir dataset, checkpoints y resultados en Google Drive.
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/ceia-vpc3"

# True = retomar desde el ultimo checkpoint de OUTPUT_DIR (corrida cortada).
RESUME = False

# Rama a clonar en Colab.
REPO_BRANCH = "main"


## 0.2 Entorno (Colab)

En Colab clona el repo (rama `REPO_BRANCH`) e instala el paquete con el extra `deep`. `torch` ya viene con CUDA en Colab. Localmente solo verifica que el paquete este instalado.

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print(f"Colab: {IN_COLAB}")


def _paquete_disponible():
    try:
        import vit_for_101_food_app  # noqa: F401

        return True
    except ImportError:
        return False


if IN_COLAB:
    REPO_URL = "https://github.com/marcoslund/ViT-for-101-food-app.git"
    REPO_DIR = "/content/ViT-for-101-food-app"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        # Ya clonado: traer la ultima version de la rama. Si actualizo codigo del paquete,
        # despues hay que reiniciar el runtime para que se reimporte.
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
    sys.path.insert(0, REPO_DIR)

    if not _paquete_disponible():
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[deep]"], check=True
        )
    os.chdir(f"{REPO_DIR}/notebooks")

    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
elif not _paquete_disponible():
    raise RuntimeError(
        "vit_for_101_food_app no esta instalado en este kernel. "
        "Elegi el kernel del .venv del proyecto (ver 'Antes de ejecutar')."
    )

print("paquete disponible:", _paquete_disponible())


## 0.3 Datos (Colab)

Lo que localmente hace `make preprocess`: descarga (o copia de Drive) el dataset, genera el split deterministico si falta y arma el cache a `CACHE_SHORT_SIDE` (288). Localmente esta celda no hace nada.

Si venias de una corrida con el cache a otra resolucion, `prep.cache` lo detecta y lo regenera; tarda unos minutos una sola vez.

In [ ]:
if IN_COLAB:
    import shutil

    from vit_for_101_food_app import config
    from vit_for_101_food_app import dataset as prep

    tar_local = config.RAW_DATA_DIR / "food-101.tar.gz"
    tar_drive = f"{DRIVE_DIR}/food-101.tar.gz"
    config.RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    if USE_DRIVE and os.path.exists(tar_drive) and not tar_local.exists():
        print("copiando el dataset desde Drive...")
        shutil.copy(tar_drive, tar_local)

    prep.download()  # descarga (~5 GB) y extrae; idempotente
    if USE_DRIVE and not os.path.exists(tar_drive):
        print("guardando el dataset en Drive para la proxima sesion...")
        shutil.copy(tar_local, tar_drive)

    if not config.TRAIN_VAL_SPLIT.exists():
        prep.split()
    prep.cache(workers=os.cpu_count() or 2)  # cache a 288px; idempotente


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch

from vit_for_101_food_app import config, plots
from vit_for_101_food_app.modeling import benchmark, evaluation, training
from vit_for_101_food_app.preprocessing import loaders

MODEL_KEY = "swin"

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
else:
    print("Dispositivo: CPU (el full training en CPU es lento; usa una GPU).")


## 1. Verificar artefactos de preprocessing

In [ ]:
id2label, label2id = benchmark.verify_artifacts()
NUM_LABELS = len(label2id)

print("cache:", config.CACHE_DIR, "| lado corto:", config.CACHE_SHORT_SIDE)
print("clases:", NUM_LABELS)


## 2. DataLoaders de Swin Transformer

`loaders.build_dataloaders("swin", ...)` lee del `AutoImageProcessor` de Swin la resolucion, la normalizacion (ImageNet) y el resample; no se escribe ningun numero a mano.

El batch lo elige `training.resolve_batch_plan` segun la resolucion del checkpoint y la GPU: modelos livianos a 224/256px entran con batch 32; un checkpoint de alta resolucion (>=320px) baja el batch, activa acumulacion de gradiente para mantener el batch efectivo en 32, y prende gradient checkpointing. Misma heuristica para todos.

In [ ]:
plan = training.resolve_batch_plan(MODEL_KEY)
print(
    "resolucion:", plan.target_size,
    "| batch:", plan.batch_size, "x", plan.grad_accum, "=", plan.effective_batch,
    "| gradient checkpointing:", plan.gradient_checkpointing,
)

dls = loaders.build_dataloaders(
    MODEL_KEY,
    train_policy="standard",
    source="cache",
    batch_size=plan.batch_size,
    num_workers=plan.num_workers,
    splits_to_load=("train", "val", "test"),
)

train_ds = dls["train"].dataset
val_ds = dls["val"].dataset
test_ds = dls["test"].dataset

print("train:", len(train_ds), "| val:", len(val_ds), "| test:", len(test_ds))
assert len(train_ds) == 68175
assert len(val_ds) == 7575
assert len(test_ds) == 25250

batch = next(iter(dls["train"]))
print("pixel_values:", tuple(batch["pixel_values"].shape), batch["pixel_values"].dtype)


## 3. Swin Transformer preentrenado

In [ ]:
CHECKPOINT = config.MODELS[MODEL_KEY]
print("checkpoint:", CHECKPOINT)

model = training.build_model(MODEL_KEY, id2label, label2id)

params = training.count_parameters(model)
print(f"parametros totales: {params['total'] / 1e6:.2f} M")
print(f"parametros entrenables: {params['trainable'] / 1e6:.2f} M")


## 4. Configuracion del fine-tuning

Misma receta para todos los modelos del benchmark (`TrainingRecipe` en `training.py`): hasta 20 epocas con early stopping (paciencia 3), mejor checkpoint por **F1 macro** en validation, warmup lineal del 5 %, `lr = 5e-4`, seed 42. El gradient checkpointing y la acumulacion de gradiente se activan solos segun la resolucion (sección 2).

In [ ]:
recipe = training.TrainingRecipe(batch_size=plan.batch_size, grad_accum_steps=plan.grad_accum)

OUTPUT_DIR, RESULTS_DIR = benchmark.default_dirs(MODEL_KEY)
if IN_COLAB and USE_DRIVE:
    OUTPUT_DIR = Path(DRIVE_DIR) / "models" / MODEL_KEY / "full"
    RESULTS_DIR = Path(DRIVE_DIR) / "results" / MODEL_KEY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

training.guard_output_dir(OUTPUT_DIR, RESUME)

print("output:", OUTPUT_DIR)
print("resultados:", RESULTS_DIR)
print(training.recipe_as_dict(recipe))


## 5. Trainer

In [ ]:
trainer = training.build_trainer(
    model,
    recipe,
    output_dir=OUTPUT_DIR,
    train_dataset=train_ds,
    val_dataset=val_ds,
    num_workers=plan.num_workers,
    gradient_checkpointing=plan.gradient_checkpointing,
)


## 6. Fine-tuning completo

In [ ]:
elapsed_train = training.run_training(trainer, resume=RESUME)
print(f"tiempo total: {elapsed_train / 3600:.2f} h")
print("best F1 macro:", trainer.state.best_metric)


## 7. Historial por epoca

In [ ]:
training.epoch_history(trainer)


In [ ]:
log_history = trainer.state.log_history
plots.history_table(log_history).to_csv(RESULTS_DIR / "training_history.csv", index=False)
rutas = plots.plot_training_curves(
    log_history,
    f"{MODEL_KEY} - {config.MODELS[MODEL_KEY]}",
    RESULTS_DIR,
    prefix=f"curvas-{MODEL_KEY}",
)


## 8. Guardar el mejor modelo

In [ ]:
BEST_DIR = training.save_best_model(trainer, OUTPUT_DIR)


## 8.1 Descargar el modelo a tu computadora

Comprime el mejor modelo y, en Colab, lo descarga al navegador. El `.zip` trae los pesos (`model.safetensors`) y la config, listos para `AutoModelForImageClassification.from_pretrained(carpeta_descomprimida)` (o para el CLI `python -m vit_for_101_food_app.modeling.predict`).

Si la descarga del navegador falla, con `USE_DRIVE = True` el modelo ya quedo en tu Drive (`OUTPUT_DIR/best`).

In [ ]:
zip_path = training.zip_model(
    BEST_DIR,
    Path("/content") / f"{MODEL_KEY}-best" if IN_COLAB else OUTPUT_DIR / f"{MODEL_KEY}-best",
)
training.download_to_browser(zip_path)


## 9. Evaluacion final sobre test

`test` se usa recien aca, una sola vez. Las predicciones se guardan **por imagen** (`predictions_test.csv`): con eso se recalcula cualquier metrica y se compara contra los otros modelos sin reentrenar.

In [ ]:
test_preds, test_metrics, elapsed_test = evaluation.evaluate_test(trainer, test_ds, id2label)
test_preds.to_csv(RESULTS_DIR / "predictions_test.csv", index=False)

print(f"tiempo test: {elapsed_test:.1f} s")
pd.Series(test_metrics)


## 10. Subset del benchmark por tercil de dificultad

La tabla que contesta la pregunta del proyecto. El subset es **el mismo archivo para todos los modelos** y `load_benchmark_subset` verifica su `sha256` antes de usarlo.

In [ ]:
subset = evaluation.load_benchmark_subset()
benchmark_tabla = evaluation.benchmark_metrics(test_preds, subset)
benchmark_tabla.to_csv(RESULTS_DIR / "benchmark_por_tercil.csv")
benchmark_tabla


## 11. Reporte por clase

In [ ]:
report_df = evaluation.per_class_report(test_preds, id2label)
report_df.to_csv(RESULTS_DIR / "report_por_clase_test.csv")

print("10 clases con peor F1")
display(report_df.head(10))
print("10 clases con mejor F1")
display(report_df.tail(10))


## 12. Confusiones mas frecuentes

In [ ]:
confusiones = evaluation.frequent_confusions(test_preds)
confusiones.to_csv(RESULTS_DIR / "confusiones_test.csv", index=False)
print("errores totales:", int((~test_preds["correct"]).sum()), "de", len(test_preds))
confusiones.head(15)


## 13. Costo arquitectonico: parametros, FLOPs, tamanio y latencia

Se mide con `modeling/evaluation.py`, el mismo codigo para todos los modelos: esa es la condicion para comparar arquitecturas.

- **Tamanio de los pesos** por precision: fp32 (lo que se entrena) y las cotas fp16/int8, que es lo que ocuparia el modelo cuantizado al enviarlo a un dispositivo.
- **Latencia** con batch 1 (una foto por vez), mediana y p90 de 100 corridas, en GPU y CPU.
- **Medicion del lado del dispositivo**: el modelo cuantizado a int8 (dinamica sobre capas `Linear`) medido en CPU — tamanio real serializado y latencia. Es lo que sostiene el eje "recursos acotados" mas alla de FLOPs/params en la GPU de entrenamiento.

In [ ]:
sample = test_ds[0]["pixel_values"].unsqueeze(0)
costo, latencia = evaluation.architectural_cost(model, sample, trainer.args.device)

print(pd.Series(costo))

# Latencia fp32 en GPU (si hay) y CPU
lat_fp32 = {k: v for k, v in latencia.items() if k in ("gpu", "cpu") and v is not None}
display(pd.DataFrame(lat_fp32))

# Medicion del lado del dispositivo: modelo cuantizado a int8 medido en CPU
print("\nCPU int8 (cuantizado):")
print(pd.Series(latencia["cpu_int8"]))


## 14. Guardar metricas

In [ ]:
metrics = benchmark.assemble_metrics(
    MODEL_KEY, recipe, trainer, elapsed_train, test_metrics, benchmark_tabla, costo, latencia
)
destino = benchmark.save_metrics(metrics, RESULTS_DIR)
print("guardado en", destino)
for p in sorted(RESULTS_DIR.iterdir()):
    print(" -", p.name)


## Resumen

- Todo el codigo de modelo/entrenamiento vive en `vit_for_101_food_app/modeling/` (`training.py`, `evaluation.py`, `benchmark.py`); esta notebook solo orquesta.
- `train` para entrenar, `val` para early stopping y seleccion de checkpoint, `test` una sola vez.
- La metrica que contesta la pregunta del proyecto es la del **subset por tercil** (sección 10).
- El mejor modelo queda en `models/swin/full/best/` y se descarga a tu compu (sección 8.1); **no** se versiona.

Con los `metrics.json` de cada modelo (mismo esquema para todos) se arma la tabla comparativa del benchmark sin reentrenar.

**En Colab** con `USE_DRIVE = True`, los resultados quedan en `ceia-vpc3/results/swin/` de tu Drive: para versionarlos, copialos a `reports/results/swin/` del checkout local.